# Notebook 1. The problem, the rule and the chain (Fig. 1) and the checks of the model

Reproduces Fig. 1, the checks of the six analytic results (Eqs. 12 to 17) and SI Fig. S1. Reads no data and writes the four panels of Fig. 1 to the folder `figs`. Run time under a minute. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

## Step 1. The ceiling and the transfer lemma

In [3]:
rng = np.random.default_rng(0)
lat = ElementModel(N=N, k=6, seed=0)
print(f'ring lattice start: H = {control_entropy(lat.strengths()):.4f} bits, ceiling {CEIL:.4f}')
m0 = lat.measures(adj=True)
print(f'lattice on the measures of the control-entropy paper: H_eig = {m0["H_adj"]:.4f}, '
      f'2^H = {2 ** m0["H_adj"]:.1f}, kappa = {m0["kappa"]:.2f}')

err = 0.0
lem_x, lem_y = [], []
for _ in range(400):
    v = rng.random(N) + 0.05
    i, j = rng.choice(N, 2, replace=False)
    eps = 1e-6
    w1, w2 = v.copy(), v.copy()
    w1[i] += eps; w1[j] -= eps
    w2[i] -= eps; w2[j] += eps
    num = (control_entropy(w1) - control_entropy(w2)) / (2 * eps)
    p = v / v.sum()
    exact = np.log2(p[j] / p[i]) / v.sum()
    err = max(err, abs(num - exact))
    lem_x.append(exact); lem_y.append(num)
print(f'A2 transfer lemma max error {err:.3e}')

ring lattice start: H = 6.6439 bits, ceiling 6.6439
lattice on the measures of the control-entropy paper: H_eig = 6.6439, 2^H = 100.0, kappa = 6.00
A2 transfer lemma max error 1.269e-09


## Step 2. Preference concentrates allocation, and it pays only when strong partners are active

In [4]:
m = ElementModel(seed=3)
for _ in range(300 if QUICK else 1500):
    m.step()

A, W = m.A, m.W
shat = m.shat()
s = m.strengths()

def alloc_entropy(i, b):
    beta = m.beta.copy(); beta[i] = b
    a = softmax_allocation(A, beta, shat)[i]
    a = a[a > 0]
    return float(-(a * np.log2(a)).sum())

p1_err = 0.0
for i in range(N):
    b = m.beta[i]
    h = 1e-5
    num = (alloc_entropy(i, b + h) - alloc_entropy(i, b - h)) / (2 * h)
    a = softmax_allocation(A, m.beta, shat)[i]
    var = (a * shat ** 2).sum() - (a * shat).sum() ** 2
    exact = -b * var / np.log(2)
    p1_err = max(p1_err, abs(num - exact))
print(f'Prop 1 allocation entropy derivative max error {p1_err:.3e}')

x = m.x.copy()
def payoff(i, b):
    beta = m.beta.copy(); beta[i] = b
    a = softmax_allocation(A, beta, shat)[i]
    return float((s[i] / s.mean()) * (a * x).sum())

p2_err = 0.0
grads, covs = [], []
for i in range(N):
    b = m.beta[i]
    h = 1e-5
    num = (payoff(i, b + h) - payoff(i, b - h)) / (2 * h)
    a = softmax_allocation(A, m.beta, shat)[i]
    cov = (a * shat * x).sum() - (a * shat).sum() * (a * x).sum()
    exact = (s[i] / s.mean()) * cov
    p2_err = max(p2_err, abs(num - exact))
    grads.append(num); covs.append(exact)
print(f'Prop 2 payoff gradient max error {p2_err:.3e}')

def mean_gradient(rho, reps=200):
    mm = ElementModel(seed=3, rho=rho)
    mm.A, mm.W, mm.beta = m.A.copy(), m.W.copy(), m.beta.copy()
    mm.c = m.c.copy(); mm.x = m.x.copy()
    a = mm.allocation(); sh = mm.shat(); st = mm.strengths()
    g = np.zeros(N)
    for _ in range(reps):
        u = mm.feedback()
        xx = mm.spread(u, a).astype(float)
        mm.x = xx
        cov = (a * sh[None, :] * xx[None, :]).sum(1) - (a * sh[None, :]).sum(1) * (a * xx[None, :]).sum(1)
        g += (st / st.mean()) * cov
    return g / reps

g_inf = mean_gradient(0.5)
g_un = mean_gradient(0.0)
print(f'rho = 0.5: mean gradient = {g_inf.mean():+.4f}')
print(f'rho = 0.0: mean gradient = {g_un.mean():+.4f}')
print(f'gradient on the essential elements, rho = 0.5: {g_inf[m.isE].mean():+.4f}, '
      f'elsewhere {g_inf[~m.isE].mean():+.4f}')

Prop 1 allocation entropy derivative max error 6.466e-11
Prop 2 payoff gradient max error 2.013e-11
rho = 0.5: mean gradient = +0.0507
rho = 0.0: mean gradient = +0.0386
gradient on the essential elements, rho = 0.5: +0.2163, elsewhere +0.0420


## Step 3. The spread is what makes strong partners more active

In [5]:
def activation_profile(pl, T=400, seed=5, spread=True):
    mm = ElementModel(seed=seed, pl=pl, spread=spread, pr=0.0, pe=0.0)
    mm.A, mm.W = m.A.copy(), m.W.copy()
    acc = np.zeros(N); sizes = []
    for _ in range(T):
        a = mm.allocation()
        u = mm.feedback(); act = mm.spread(u, a)
        mm.x = act.astype(float); acc += mm.x
        sizes.append(mm.n_act / N)
    return acc / T, np.mean(sizes)

sh_now = m.shat()
print('p_l    mean activation  corr(strength, activation)')
pl_grid = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
prof = {}
for pl in pl_grid:
    acc, size = activation_profile(pl)
    r = np.corrcoef(sh_now, acc)[0, 1] if acc.std() > 1e-12 else float('nan')
    prof[pl] = acc
    print(f'{pl:<6.1f} {size:<16.3f} {r:+.3f}')
acc_off, size_off = activation_profile(0.3, spread=False)
r_off = np.corrcoef(sh_now, acc_off)[0, 1]
print(f'seeding only (no spread): mean activation {size_off:.3f}, corr {r_off:+.3f}')

p_l    mean activation  corr(strength, activation)
0.0    0.092            -0.071
0.1    0.133            +0.104
0.2    0.207            +0.432
0.3    0.321            +0.628
0.5    0.610            +0.670
0.7    0.789            +0.533
seeding only (no spread): mean activation 0.094, corr -0.072


## Step 4. Imitation turns variation into direction

In [6]:
def exact_drift(model):
    tot = 0.0
    for i in range(N):
        nb = np.flatnonzero(model.A[i])
        tot += np.mean(fermi(model.c[nb] - model.c[i], model.p['s']) * (model.beta[nb] - model.beta[i]))
    return model.p['pr'] * tot / N

def mc_drift(model, rounds):
    rng2 = np.random.default_rng(11)
    out = np.empty(rounds)
    for r in range(rounds):
        b = model.beta.copy()
        for i in np.flatnonzero(rng2.random(N) < model.p['pr']):
            nb = np.flatnonzero(model.A[i])
            j = nb[rng2.integers(nb.size)]
            if rng2.random() < fermi(model.c[j] - model.c[i], model.p['s']):
                b[i] = model.beta[j]
        out[r] = b.mean() - model.beta.mean()
    return out

R = 1000 if QUICK else 4000
ex = exact_drift(m)
mc = mc_drift(m, R)
z = (mc.mean() - ex) / (mc.std(ddof=1) / np.sqrt(R))
print(f'Prop 4 exact drift {ex:+.6f}, Monte Carlo {mc.mean():+.6f} +/- {mc.std(ddof=1)/np.sqrt(R):.6f}')
print(f'Prop 4 exact drift vs Monte Carlo (z score) {z:.2f}')

flat = ElementModel(seed=7)
flat.beta[:] = 0.4
print(f'drift with every preference equal: {exact_drift(flat):+.2e}')

Prop 4 exact drift -0.001068, Monte Carlo -0.000981 +/- 0.000160
Prop 4 exact drift vs Monte Carlo (z score) 0.54
drift with every preference equal: +0.00e+00


## Step 5. Preference lowers control entropy, and commitment holds what it built

### The drawing style of the SI figures

In [7]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

SI_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.6, 'xtick.labelsize': 6.0,
         'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
         'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False,
         'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'}
INK, NEUTRAL = '#1A1A1A', '#37474F'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
SI_COND = {'baseline': '#C4C9CD', 'explore': '#8A8A8A', 'imitate': '#5F6B72', 'both': NAVY}
SI_LABEL = {'baseline': 'baseline', 'explore': 'exploration', 'imitate': 'imitation', 'both': 'both'}
_SI_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_SI_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_SI_COOL, _SI_HOT]))
HOT3 = LinearSegmentedColormap.from_list('hot_half', CMAP3(np.linspace(0.50, 0.92, 256)))
SI_VLIM = 2.7157337518864986                       # the scale of Fig. 2 and of ref. 3, in bits
SI_CNORM = Normalize(-SI_VLIM, SI_VLIM)
DOT = (0, (1.2, 1.4))


def si_share_colour(w):
    # an element colored by its share against an even share, as in Figs. 2 and 3
    w = np.asarray(w, float); b = np.log2(w / w.sum() * len(w))
    return CMAP3(SI_CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def si_figure(w, h, label):
    plt.rcParams.update(SI_RC)
    fig = plt.figure(figsize=(w, h)); fig.set_label(label)
    return fig


def si_axes(fig, l, b, w, h):
    W, H = fig.get_size_inches()
    return fig.add_axes([l / W, b / H, w / W, h / H])


def si_head(fig, ax, letter, title, dy=0.12):
    # the title centered over the plot, the letter in bold beside its left end
    W, H = fig.get_size_inches()
    bb = ax.get_position()
    y = bb.y1 + dy / H
    t = fig.text((bb.x0 + bb.x1) / 2, y, title, fontsize=7.6, ha='center', va='bottom')
    ext = t.get_window_extent(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())
    fig.text(ext.x0 - 0.06 / W, y, letter, fontsize=9.0, fontweight='bold', ha='right', va='bottom')


def si_ticks(lo, hi):
    return [lo, (lo + hi) / 2, hi]


def si_nice(lo, hi, zero=False):
    # three round ticks, minimum, middle and maximum, as tight as round numbers allow around the data
    lo, hi = (min(0.0, lo), max(0.0, hi)) if zero else (lo, hi)
    span = (hi - lo) or abs(hi) or 1.0
    for e in range(int(np.floor(np.log10(span))) - 2, int(np.floor(np.log10(span))) + 2):
        for m in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8):
            u = m * 10.0 ** e
            a = np.floor(lo / u + 1e-9) * u
            b = np.ceil(hi / u - 1e-9) * u
            n = int(round((b - a) / u))
            if n % 2:
                b += u; n += 1
            if n <= 2:
                return [round(a, 10), round(a + u, 10), round(b, 10)]
    return [lo, (lo + hi) / 2, hi]


def si_frame(ax, xt=None, yt=None, xlog=False, ylog=False, xfmt=None, yfmt=None, grid=True):
    # a line above and below, none at the sides, ticks at the given values, a dotted rule at the middle tick
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    g = lambda v, _: f'{v:,g}'
    if xt is not None:
        ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(xfmt or g))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(yfmt or g))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        if grid:
            for v in yt[1:-1]:
                ax.axhline(v, color='gray', alpha=0.45, ls=':', lw=0.5, zorder=0)
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)
    ax.set_axisbelow(True)


def si_cbar(fig, l, b, w, h, cmap, lo, hi, label, fmt='{:g}'):
    ax = si_axes(fig, l, b, w, h)
    ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=cmap, aspect='auto', extent=[0, 1, lo, hi])
    ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([fmt.format(lo), fmt.format(hi)])
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
    ax.set_ylabel(label, fontsize=6.0, labelpad=1.5)
    return ax


def si_bars(ax, groups, colors, labels, dots=True):
    # bars at the median of each group with every run as a dot beside it
    for i, v in enumerate(groups):
        v = np.asarray(v, float)
        ax.bar(i, np.median(v), color=colors[i], width=0.58, zorder=2)
        if dots:
            ax.plot(np.full(len(v), i) + 0.20 + np.linspace(-0.05, 0.05, len(v)), v, '.', ms=1.8, color=INK,
                    alpha=0.6, zorder=3)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(labels)
    ax.set_xlim(-0.6, len(groups) - 0.4)


def si_bracket(ax, i, j, y, h, text):
    ax.plot([i, i, j, j], [y, y + h, y + h, y], '-', lw=0.6, color=INK)
    ax.text((i + j) / 2, y + h, text, ha='center', va='bottom', fontsize=6.4)


def si_radial(A, n_min=2, dmax=None):
    # elements at the angle of their place on the ring, drawn inward by degree, as in Fig. 5C
    d = A.sum(1).astype(float); n = len(d)
    th = np.pi / 2 - 2 * np.pi * np.arange(n) / n
    dmax = max(d.max() if dmax is None else dmax, n_min + 1)
    r = np.clip(np.log(dmax / np.maximum(d, n_min)) / np.log(dmax / n_min), 0, 1)
    return np.c_[r * np.cos(th), r * np.sin(th)], d


def si_structure(ax, A, W=None, isE=None, s=None, lim=1.12, dmax=None):
    # a learned structure: couplings darker and wider with weight, elements colored by their share of the
    # coupling against an even share and sized by degree, the task-essential elements ringed
    from matplotlib.collections import LineCollection
    xy, d = si_radial(A, dmax=dmax)
    iu = np.array(np.where(np.triu(A))).T
    if W is not None:
        s = (W * A).sum(1) if s is None else s
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / max(ww.max() - ww.min(), 1e-12))
    else:
        x = np.full(len(iu), 0.3)
    o = np.argsort(x, kind='stable')
    ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.25 + 0.55 * x[o]),
                                     linewidths=0.12 + 0.8 * x[o], alpha=0.75, zorder=1))
    size = 3.0 + 110.0 * ((d - 1) / (len(d) - 2)) ** 1.15
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=si_share_colour(s)[o], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    if isE is not None:
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 2.6) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.6, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal'); ax.axis('off')

In [8]:
rng3 = np.random.default_rng(2)
A_l = ring_lattice(N, 6)
W_l = A_l * (1.0 + 0.3 * rng3.random((N, N)))
W_l = np.triu(W_l, 1); W_l = W_l + W_l.T
s_l = W_l.sum(1); sh_l = s_l / s_l.max()
k_l = A_l.sum(1)

def compression_at(b):
    a = softmax_allocation(A_l, np.full(N, b), sh_l)
    q = a.mean(0)
    return CEIL - control_entropy(q)

qprime = np.zeros(N)
for j in range(N):
    nbj = np.flatnonzero(A_l[j])
    qprime[j] = np.sum((sh_l[j] - np.array([sh_l[np.flatnonzero(A_l[i])].mean() for i in nbj])) / k_l[nbj]) / N
pred = lambda b: N * b ** 2 / (2 * np.log(2)) * (qprime ** 2).sum()
ratios = []
bs = [0.25, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
for b in bs:
    ratios.append(compression_at(b) / pred(b))
print(f'Prop 5 ratio observed/predicted (min, max) {min(ratios):.4f} to {max(ratios):.4f}')

eq = ElementModel(seed=4, pr=0.0, pe=0.0, f0=0.2)
for _ in range(200 if QUICK else 800):
    eq.step()
C_eq = eq.centrality()
f_eq = commitment(eq.phi0, eq.phi1, C_eq, eq.p['fm'])
P_eq = protection(f_eq, eq.c, eq.p['pmax'], eq.p['chalf'])
print(f'Prop 6 mean commitment {f_eq.mean():.3f}, mean protection {P_eq.mean():.3f}, '
      f'mean excess weight {np.mean(eq.W[eq.A] - 1.0):.3f}')
eq0 = ElementModel(seed=4, pr=0.0, pe=0.0, f0=0.0)
for _ in range(200 if QUICK else 800):
    eq0.step()
print(f'Prop 6 without commitment: mean excess weight {np.mean(eq0.W[eq0.A] - 1.0):.3f}, '
      f'strength ratio committed/uncommitted {eq.strengths().mean()/eq0.strengths().mean():.2f}')

fig = si_figure(6.93, 3.55, 'SI Fig S1')
PW, PH = 1.62, 0.92
XS = [0.42, 2.78, 5.14]
YS = [2.20, 0.45]

def panel(i, j):
    return si_axes(fig, XS[j], YS[i], PW, PH)

ax = panel(0, 0)
t = si_nice(min(lem_x), max(lem_x))
ax.plot(t[::2], t[::2], '-', lw=0.7, color=NEUTRAL, zorder=1)
ax.plot(lem_x, lem_y, '.', ms=2.4, color=NAVY, zorder=2)
si_frame(ax, t, t)
ax.set_xlabel(r'$\log_2(p_j/p_i)\,/\,\Sigma s$', labelpad=1); ax.set_ylabel('finite difference', labelpad=1)
si_head(fig, ax, 'A', 'Transfer lemma')

ax = panel(0, 1)
bgrid = np.linspace(-3, 3, 61)
he = np.array([alloc_entropy(0, b) for b in bgrid])
ax.plot(bgrid, he, '-', lw=1.1, color=GREEN)
si_frame(ax, [-3, 0, 3], si_nice(he.min(), he.max()))
ax.set_xlabel(r'preference $\beta_i$', labelpad=1); ax.set_ylabel('allocation entropy (bits)', labelpad=1)
si_head(fig, ax, 'B', 'Preference concentrates allocation')

ax = panel(0, 2)
t = si_nice(min(min(covs), min(grads)), max(max(covs), max(grads)))
ax.plot(t[::2], t[::2], '-', lw=0.7, color=NEUTRAL, zorder=1)
ax.plot(covs, grads, '.', ms=2.4, color=RED, zorder=2)
si_frame(ax, t, t)
ax.set_xlabel('covariance formula', labelpad=1); ax.set_ylabel('numerical gradient', labelpad=1)
si_head(fig, ax, 'C', 'Payoff gradient')

ax = panel(1, 0)
o = np.argsort(sh_now)
for pl, c in zip([0.0, 0.1, 0.3, 0.7], [CMAP3(0.12), CMAP3(0.32), CMAP3(0.72), CMAP3(0.86)]):
    ax.plot(sh_now[o], prof[pl][o], '.', ms=2.0, color=c, label=f'{pl:g}')
si_frame(ax, [0, 0.5, 1], [0, 0.5, 1])
ax.legend(title=r'$p_l$', title_fontsize=6, loc='center right', bbox_to_anchor=(1.0, 0.40), ncol=2,
          handletextpad=0.1, labelspacing=0.2, columnspacing=0.8, markerscale=2.2)
ax.set_xlabel('strength, scaled', labelpad=1); ax.set_ylabel('activation rate', labelpad=1)
si_head(fig, ax, 'D', 'Spread favors strong elements')

ax = panel(1, 1)
t = si_nice(min(mc.min(), ex), max(mc.max(), ex))
ax.hist(mc, bins=np.linspace(t[0], t[-1], 41), color='#B7C0C7', lw=0)
ax.axvline(ex, color=RED, lw=1.0)
cnt = np.histogram(mc, bins=np.linspace(t[0], t[-1], 41))[0].max()
si_frame(ax, t, si_nice(0, cnt, zero=True), xfmt=lambda v, _: f'{v:g}')
ax.text(ex, si_nice(0, cnt, zero=True)[-1] * 0.96, ' exact', color=RED, fontsize=5.8, ha='left', va='top')
ax.set_xlabel('change of the mean preference', labelpad=1); ax.set_ylabel('rounds', labelpad=1)
si_head(fig, ax, 'E', 'Imitation drift')

ax = panel(1, 2)
comp = np.array([compression_at(b) for b in bs]); prd = np.array([pred(b) for b in bs])
ax.plot(bs, prd, '--', lw=0.9, color=NEUTRAL, label='second order')
ax.plot(bs, comp, 'o', ms=2.6, color=NAVY, label='computed')
si_frame(ax, [0, 1.5, 3], si_nice(0, max(comp.max(), prd.max()), zero=True))
ax.legend(loc='upper left', handlelength=1.6)
ax.set_xlabel(r'common preference $\beta$', labelpad=1); ax.set_ylabel('compression (bits)', labelpad=1)
si_head(fig, ax, 'F', 'Compression from the ceiling')
plt.show()

Prop 5 ratio observed/predicted (min, max) 0.9956 to 0.9998
Prop 6 mean commitment 0.200, mean protection 0.637, mean excess weight 2.950
Prop 6 without commitment: mean excess weight 0.827, strength ratio committed/uncommitted 2.16


## Step 6. Checks of the analytic results

In [9]:
ok = {}
ok['A2 transfer lemma below 1e-4'] = err < 1e-4
ok['Prop 1 below 1e-8'] = p1_err < 1e-8
ok['Prop 2 below 1e-8'] = p2_err < 1e-8
ok['Prop 3 activation rises with strength when p_l > 0'] = np.corrcoef(sh_now, prof[0.3])[0, 1] > 0.3
ok['Prop 3 activation unrelated to strength without the spread'] = abs(r_off) < 0.2
ok['Prop 4 |z| below 2'] = abs(z) < 2
ok['Prop 5 ratio within 1 +/- 0.01'] = max(abs(np.array(ratios) - 1)) < 0.01
ok['Prop 6 protection raises the excess weight'] = np.mean(eq.W[eq.A] - 1.0) > np.mean(eq0.W[eq0.A] - 1.0)
ok['lattice sits at the ceiling'] = abs(control_entropy(lat.strengths()) - CEIL) < 1e-9
for kk, vv in ok.items():
    print(f'{"PASS" if vv else "FAIL"}  {kk}')
print('all checks passed:', all(ok.values()))

PASS  A2 transfer lemma below 1e-4
PASS  Prop 1 below 1e-8
PASS  Prop 2 below 1e-8
PASS  Prop 3 activation rises with strength when p_l > 0
PASS  Prop 3 activation unrelated to strength without the spread
PASS  Prop 4 |z| below 2
PASS  Prop 5 ratio within 1 +/- 0.01
PASS  Prop 6 protection raises the excess weight
PASS  lattice sits at the ceiling
all checks passed: True


In [10]:
import os
from matplotlib.transforms import Bbox

PANEL_DIR = os.environ.get('FIG_DIR', 'figs')


def save_panels(fig, stem, regions, hide=()):
    # every region of the drawn figure saved as its own panel file, PNG at 300 dpi and PDF and SVG with editable
    # text; regions maps a panel name to its box (x0, y0, x1, y1) in inches from the lower left corner of the
    # figure, and the artists in hide, which join the panels, are left out of the panel files
    os.makedirs(PANEL_DIR, exist_ok=True)
    for a in hide:
        a.set_visible(False)
    with plt.rc_context({'pdf.fonttype': 42, 'svg.fonttype': 'none'}):
        for name, box in regions.items():
            path = os.path.join(PANEL_DIR, f'{stem}{name}')
            kw = dict(bbox_inches=Bbox.from_extents(*box), facecolor='white')
            fig.savefig(path + '.png', dpi=300, metadata={'Software': None}, **kw)
            fig.savefig(path + '.pdf', metadata={'Creator': None, 'Producer': None, 'CreationDate': None}, **kw)
            fig.savefig(path + '.svg', metadata={'Creator': None, 'Date': None}, **kw)
            svg = open(path + '.svg', encoding='utf-8').read().replace('id="matplotlib.', 'id="')
            open(path + '.svg', 'w', encoding='utf-8').write(svg)
    for a in hide:
        a.set_visible(True)
    print('panel files:', ', '.join(f'{stem}{n}' for n in regions))

## Step 7. Figure 1. Four accounts in four quadrants, and the game the elements play

In [11]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Circle, Polygon, FancyBboxPatch, Arc, Ellipse, Rectangle

# the style of Figs. 2 to 5: one font, black headings with a bold letter, and one meaning per color.
# gold the stimulus and the payoff it becomes, green exploration and perception, red copying and action,
# blue the coupling, gray the structure, and the share of coupling on the scale of Fig. 2
plt.rcdefaults()                                   # the same drawing in this notebook and on its own
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.linewidth': 0.6,
                     'mathtext.fontset': 'dejavusans', 'figure.dpi': 110})
INK, NEUTRAL, LINK = '#1A1A1A', '#37474F', '#C3CBD0'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
GOLD_T, PALE, LIGHT = '#8C6D0E', '#EAF0F8', '#B4B9BD'
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
CNORM = Normalize(-2.7157337518864986, 2.7157337518864986)

def fermi_curve(dc, s):
    return 1.0 / (1.0 + np.exp(-s * dc))

def colour(w, n):
    b = np.log2(np.asarray(w, float) * n)
    return CMAP(CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))

U = 0.5                                            # inches per unit of the drawing
W, H = 13.86, 7.56                                 # the width of Figs. 2 to 5
fig = plt.figure(figsize=(W * U, H * U)); fig.set_label('Manuscript Fig 1')
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.set_aspect('equal'); ax.axis('off')

def arrow(xy, xytext, color=NEUTRAL, lw=0.8, style='-|>', rad=0.0, ms=7, ls='-', z=5):
    ax.annotate('', xy=xy, xytext=xytext, zorder=z,
                arrowprops=dict(arrowstyle=style, lw=lw, color=color, mutation_scale=ms, linestyle=ls,
                                connectionstyle=f'arc3,rad={rad}', shrinkA=0, shrinkB=0))

def label(x, y, s, **k):
    kw = dict(fontsize=6.0, ha='center', va='center', color=INK); kw.update(k); return ax.text(x, y, s, **kw)

def heading(x0, x1, ytop, letter, title, sub):
    # as in Figs. 2 to 5: the bold letter at the left, the title centered over the panel, the subtitle beneath
    ax.text(x0 + 0.1, ytop - 0.38, letter, fontsize=9.4, fontweight='bold', ha='left', va='bottom')
    ax.text((x0 + x1) / 2, ytop - 0.38, title, fontsize=7.9, ha='center', va='bottom')
    ax.text((x0 + x1) / 2, ytop - 0.42, sub, fontsize=6.2, ha='center', va='top')

def dotbox(x, y, w, h):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.1', facecolor='none',
                                edgecolor=NEUTRAL, lw=0.7, linestyle=(0, (1.0, 1.6)), zorder=1))

XC, YC = 6.74, 3.86                                # the junction of the four accounts
TOP, MID = H, YC - 0.12                            # the tops of the upper and the lower panels

# ================================================================== A. reflex chain (top left)
heading(0.0, XC - 0.1, TOP, 'A', 'Reflex chain', 'S–R, the conductor left closed')
Y1 = 5.45
ax.plot([1.05, 0.88, 0.88, 1.05], [Y1 + 0.42, Y1 + 0.42, Y1 - 0.42, Y1 - 0.42], color=NEUTRAL, lw=0.7)
for dy in (-0.24, 0.0, 0.24):
    ax.plot([0.88, 1.18], [Y1 + dy, Y1], color=NEUTRAL, lw=0.5)
ax.plot([1.18, 4.65], [Y1, Y1], color=NEUTRAL, lw=1.0)
ax.add_patch(Polygon([[2.85, Y1 + 0.18], [2.85, Y1 - 0.18], [2.55, Y1]], closed=True, facecolor='white',
                     edgecolor=NEUTRAL, lw=0.7, zorder=3))
ax.add_patch(Circle((3.06, Y1), 0.11, facecolor='white', edgecolor=NEUTRAL, lw=0.7, zorder=3))
dotbox(1.5, Y1 - 0.44, 3.5, 0.88)
label(5.0, Y1 + 0.5, 'black box', fontsize=5.4, color=GREY, style='italic', ha='right', va='bottom')
ax.plot([4.65, 4.85, 4.65], [Y1 + 0.16, Y1, Y1 - 0.16], color=NEUTRAL, lw=0.7)
ax.add_patch(Ellipse((5.5, Y1), 0.52, 1.25, facecolor=PALE, edgecolor=NEUTRAL, lw=0.7))
for ddx in (-0.13, 0.0, 0.13):
    hh = 0.5 * np.sqrt(max(0.0, 1 - (ddx / 0.26) ** 2))
    ax.plot([5.5 + ddx] * 2, [Y1 - hh, Y1 + hh], color=LIGHT, lw=0.45)
ax.plot([5.5, 5.5], [Y1 + 0.62, Y1 + 0.8], color=NEUTRAL, lw=0.6)
ax.plot([5.5, 5.5], [Y1 - 0.62, Y1 - 0.8], color=NEUTRAL, lw=0.6)
label(0.98, Y1 - 0.6, 'receptor', va='top'); label(2.8, Y1 - 0.2, 'conductor', va='top')
label(5.5, Y1 - 0.9, 'effector', va='top')
label(0.5, Y1 + 0.62, 'stimulus', color=GOLD_T, fontsize=5.8, ha='left')
label(5.5, Y1 + 0.93, 'response', color=GREY, fontsize=5.8)

# ================================================================== B. processing hierarchy (bottom left)
heading(0.0, XC - 0.1, MID, 'B', 'Processing hierarchy', 'the box opened, stages, memories and feedback')
YB = 1.95
def rbox(x, y, w, h, s, fs=5.4):
    ax.add_patch(Rectangle((x, y), w, h, facecolor=PALE, edgecolor=BLUE, lw=0.6, zorder=3))
    label(x + w / 2, y + h / 2, s, fontsize=fs, zorder=4)
label(0.5, YB + 0.62, 'stimulus', color=GOLD_T, fontsize=5.8, ha='left')
ax.add_patch(Rectangle((0.98, YB - 0.33), 0.48, 0.66, facecolor='white', edgecolor=NEUTRAL, lw=0.6, zorder=3))
label(1.22, YB - 0.42, 'sensory\nstore', fontsize=5.0, color=GREY, va='top')
for dy in (-0.16, 0.0, 0.16):
    arrow((1.75, YB + dy), (0.7, YB + dy), lw=0.5, ms=4, z=4)
rbox(1.75, YB - 0.3, 0.95, 0.6, 'perception')
arrow((2.95, YB), (2.7, YB), lw=0.7, ms=5)
rbox(2.95, YB - 0.3, 1.2, 0.6, 'response\nselection')
arrow((4.4, YB), (4.15, YB), lw=0.7, ms=5)
rbox(4.4, YB - 0.3, 1.2, 0.6, 'response\nexecution')
arrow((6.1, YB), (5.6, YB), lw=0.7, ms=5)
label(5.62, YB + 0.42, 'response', color=GREY, fontsize=5.8, ha='left', va='bottom')
ax.add_patch(Ellipse((4.45, 3.0), 1.3, 0.34, facecolor='white', edgecolor=LIGHT, lw=0.5, zorder=3))
label(4.45, 3.0, 'attention', fontsize=5.0, color=GREY, zorder=4)
for x_ in (2.22, 3.55, 5.0):
    arrow((x_, YB + 0.32), (4.45 + 0.5 * (x_ - 4.45) / 2.2, 2.84), color=LIGHT, lw=0.5, ms=4)
ax.plot([3.9, 2.82, 2.82, 3.2], [2.86, 2.58, 1.62, 1.62], color=LIGHT, lw=0.5, zorder=2)
arrow((3.2, 1.45), (3.2, 1.62), color=LIGHT, lw=0.5, ms=4)
def lbox(x, y, w, h, s):
    ax.add_patch(Rectangle((x, y), w, h, facecolor='#F6F6F6', edgecolor=LIGHT, lw=0.5, zorder=3))
    label(x + w / 2, y + h / 2, s, fontsize=4.9, color=GREY, zorder=4)
lbox(1.75, 0.98, 0.95, 0.47, 'long-term\nmemory'); lbox(3.1, 0.98, 0.95, 0.47, 'working\nmemory')
arrow((2.22, YB - 0.32), (2.22, 1.45), color=LIGHT, lw=0.5, ms=4)
arrow((3.45, 1.45), (3.45, YB - 0.32), color=LIGHT, lw=0.5, ms=4)
arrow((3.68, YB - 0.32), (3.68, 1.45), color=LIGHT, lw=0.5, ms=4)
arrow((2.7, 1.31), (3.1, 1.31), color=LIGHT, lw=0.5, ms=4); arrow((3.1, 1.12), (2.7, 1.12), color=LIGHT, lw=0.5, ms=4)
# feedback from the response back to the stimulus, drawn as in the source: down, back along the bottom, up
ax.plot([5.9, 5.9, 0.72, 0.72], [YB, 0.62, 0.62, YB - 0.2], color=GOLD, lw=0.8, zorder=2)
arrow((3.2, 0.62), (3.4, 0.62), color=GOLD, lw=0.8, ms=5)
arrow((0.72, YB - 0.02), (0.72, YB - 0.2), color=GOLD, lw=0.8, ms=5)
label(3.3, 0.52, 'feedback', fontsize=5.6, color=GOLD_T, va='top', style='italic')

# ================================================================== D. element-level game (top right)
heading(XC + 0.1, W, TOP, 'D', 'Element-level game', 'capability, consolidation and the Fermi rule')
xi, xj, yn = 8.05, 12.45, 5.3
x0 = 0.5 * (xi + xj)
fx0, fx1, fy0, fy1 = xi + 0.5, xj - 0.5, yn - 0.62, yn + 0.62
def fx(dc): return fx0 + (dc + 3.0) / 6.0 * (fx1 - fx0)
def fy(p): return fy0 + p * (fy1 - fy0)
for p in (0, 0.5, 1):
    ax.plot([fx0, fx1], [fy(p)] * 2, lw=0.4, color='#D3D3D3', zorder=0, ls=':' if p == 0.5 else '-')
ax.plot([fx(0)] * 2, [fy0, fy1], lw=0.4, color='#D3D3D3', zorder=0)
dc = np.linspace(-3, 3, 240)
for sv, c in [(0.5, LIGHT), (2.0, RED), (8.0, NAVY)]:
    ax.plot(fx(dc), fy(fermi_curve(dc, sv)), '-', lw=0.9, color=c, alpha=0.75, zorder=1, solid_capstyle='round')
label(fx(0) + 0.06, fy(1) - 0.03, '1', fontsize=5.0, ha='left', va='top', color=GREY)
label(fx(0) - 0.06, fy(0) + 0.03, '0', fontsize=5.0, ha='right', va='bottom', color=GREY)
label(fx(-3) + 0.04, fy(0.5) - 0.08, '$-3$', fontsize=5.0, ha='left', va='top', color=GREY)
label(fx(3) - 0.04, fy(0.5) - 0.08, '3', fontsize=5.0, ha='right', va='top', color=GREY)
label(fx0 + 0.05, fy1 - 0.02, '$s$ = 0.5, 2, 8', fontsize=5.2, ha='left', va='top', color=GREY)
label(fx1 - 0.05, fy(0.16), 'imitation\nEq. 10', fontsize=5.4, ha='right', color=RED, style='italic')
label(x0 + 0.55, fy0 - 0.3, r'$p = 1\,/\,(1 + e^{-s(c_j - c_i)})$', fontsize=5.2, ha='left')
label(x0 + 0.55, fy0 - 0.55, '$c_j - c_i$, capability difference', fontsize=5.2, ha='left')

def neuron(x, y, core, lab, flip=False):
    # the element of Figs. 4 and 5, a white disc whose gold core is its capability, with dendrites
    for ang in ((150, 180, 210) if not flip else (-30, 0, 30)):
        t = np.radians(ang)
        x1, yy1, x2, yy2 = x + 0.31 * np.cos(t), y + 0.31 * np.sin(t), x + 0.66 * np.cos(t), y + 0.66 * np.sin(t)
        ax.plot([x1, x2], [yy1, yy2], color=NEUTRAL, lw=0.8, solid_capstyle='round', zorder=3)
        for sub in (-22, 22):
            t2 = np.radians(ang + sub)
            ax.plot([x2, x2 + 0.18 * np.cos(t2)], [yy2, yy2 + 0.18 * np.sin(t2)], color=NEUTRAL, lw=0.6, zorder=3)
    ax.add_patch(Circle((x, y), 0.31, facecolor='white', edgecolor=NEUTRAL, lw=0.8, zorder=4))
    ax.add_patch(Circle((x, y), core, facecolor=GOLD, edgecolor='none', zorder=5))
    ax.text(x + (0.2 if flip else -0.2), y + 0.46, lab, fontsize=7.6, style='italic',
            ha='left' if flip else 'right', va='bottom', zorder=6)

neuron(xi, yn, 0.12, 'i'); neuron(xj, yn, 0.2, 'j', flip=True)
bx0 = x0 - 0.5
ax.plot([xi + 0.31, bx0 + 0.02], [yn, yn], color=BLUE, lw=1.4, solid_capstyle='round', zorder=3)
ax.add_patch(Ellipse((bx0 + 0.17, yn), 0.36, 0.44, facecolor=BLUE, alpha=0.9, edgecolor='none', zorder=4))
for (ddx, ddy) in ((0.10, 0.09), (0.17, -0.06), (0.07, -0.11), (0.22, 0.11), (0.14, 0.0), (0.24, -0.03)):
    ax.add_patch(Circle((bx0 + ddx, yn + ddy), 0.034, facecolor='white', edgecolor='none', zorder=5))
ax.add_patch(Rectangle((bx0 + 0.42, yn - 0.25), 0.10, 0.50, facecolor='#E6E6E6', edgecolor='none', zorder=3))
for ddy in (-0.15, 0.0, 0.15):
    ax.add_patch(Rectangle((bx0 + 0.52, yn + ddy - 0.045), 0.10, 0.09, facecolor=NAVY, edgecolor='none', zorder=4))
ax.plot([bx0 + 0.62, xj - 0.31], [yn, yn], color=BLUE, lw=1.4, solid_capstyle='round', zorder=3)
label(x0, yn + 0.7, 'consolidate  $w_{ij}$', fontsize=6.2, va='bottom', color=BLUE)
for x_, h, lab in ((xi, 0.28, 'c_i'), (xj, 0.62, 'c_j')):
    ax.add_patch(Rectangle((x_ - 0.12, yn + 0.44), 0.24, h, facecolor=GOLD, alpha=0.9, edgecolor='none'))
    label(x_, yn + 0.44 + h + 0.04, f'${lab}$', fontsize=6.8, va='bottom')
label(x0, yn + 1.06, 'functional capability', fontsize=5.6, color=GOLD_T)
for x_ in (xi, xj):
    arrow((x_, yn - 0.36), (x_, yn - 0.9), color=GOLD, lw=1.1, ms=7)
label(xi - 0.18, yn - 0.92, 'stimulus', fontsize=5.6, color=GOLD_T, ha='right')
label(xj + 0.18, yn - 0.92, 'stimulus', fontsize=5.6, color=GOLD_T, ha='left')
label(xi - 0.18, yn - 0.64, r'explore $\beta_i$', fontsize=5.8, color=GREEN, ha='right')

# ================================================================== C. self-organized search (bottom right)
heading(XC + 0.1, W, MID, 'C', 'Self-organized search', 'constraints, the perception–action cycle, the structure left')
Y3 = 1.72
cx, cy, R = 10.3, Y3, 0.85
vx, vy, vr = 7.95, Y3, 0.42
for (dx, dy, lab, la) in ((0.0, 0.35, 'T', (0, 0.57)), (-0.3, -0.19, 'I', (-0.53, -0.34)),
                          (0.3, -0.19, 'E', (0.53, -0.34))):
    ax.add_patch(Circle((vx + dx, vy + dy), vr, facecolor='#DDE3E8', alpha=0.55, edgecolor=NEUTRAL, lw=0.5))
    label(vx + la[0], vy + la[1], lab, fontsize=6.2)
label(vx, vy, 'M', fontsize=6.2, fontweight='bold')
label(vx, vy + 0.84, 'mobility  stability  manipulation', fontsize=4.9, color=GREY, va='bottom')
label(vx - 0.5, vy - 0.66, 'cognition\nperception\naction', fontsize=4.9, color=GREY, va='top')
label(vx + 0.6, vy - 0.66, 'regulatory\nnonregulatory', fontsize=4.9, color=GREY, va='top')
arrow((cx - R - 0.1, Y3), (vx + 0.84, Y3))
ax.add_patch(Circle((cx, cy), R, facecolor='#F5F7F8', edgecolor='none'))
for th0, th1 in ((12, 168), (192, 348)):
    ax.add_patch(Arc((cx, cy), 2 * R, 2 * R, theta1=th0, theta2=th1, color=NEUTRAL, lw=1.1))
for th in (168, 348):
    t, t2 = np.radians(th), np.radians(th - 8)
    arrow((cx + R * np.cos(t), cy + R * np.sin(t)), (cx + R * np.cos(t2), cy + R * np.sin(t2)), lw=1.1, ms=6)
label(cx, cy + R + 0.07, 'perception', fontsize=6.0, va='bottom', color=GREEN)
label(cx, cy - R - 0.07, 'action', fontsize=6.0, va='top', color=RED)
for k in range(8):
    t = np.radians(45 * k); col = GREEN if k % 2 == 0 else RED
    arrow((cx + 0.5 * np.cos(t), cy + 0.5 * np.sin(t)), (cx + 0.13 * np.cos(t), cy + 0.13 * np.sin(t)),
          color=col, lw=0.8, ms=4.5)
    ax.add_patch(Circle((cx + 0.57 * np.cos(t), cy + 0.57 * np.sin(t)), 0.048, facecolor=col, edgecolor='none',
                        zorder=4))
ax.add_patch(Circle((cx, cy), 0.11, facecolor=INK, edgecolor='none', zorder=4))
label(cx, cy - R - 0.36, 'search of the workspace', fontsize=5.6, va='top', color=GREY, style='italic')
# the structure the search leaves, colored by its share of the coupling and sized by degree as in Fig. 2
G = nx.barabasi_albert_graph(16, 1, seed=7); pos = nx.kamada_kawai_layout(G)
deg = np.array([G.degree(n) for n in G.nodes()]); sx, sy, sc = 12.75, Y3, 0.85
for u_, v_ in G.edges():
    ax.plot([sx + sc * pos[u_][0], sx + sc * pos[v_][0]], [sy + sc * pos[u_][1], sy + sc * pos[v_][1]],
            color=LINK, lw=0.8, zorder=2)
ax.scatter([sx + sc * pos[n][0] for n in G.nodes()], [sy + sc * pos[n][1] for n in G.nodes()],
           s=7 + 9 * deg, c=colour(deg / deg.sum(), len(deg)), edgecolors=NEUTRAL, linewidths=0.4, zorder=3)
arrow((sx - 0.98, Y3), (cx + R + 0.1, Y3))
label(sx, Y3 - 0.95, 'coordinative\nstructure', fontsize=6.0, va='top')

# ================================================================== the junction: a dotted cross behind the four accounts
_before = set(ax.get_children())
JD, JA = 0.08, 0.45
JK = dict(color=NEUTRAL, lw=0.5, alpha=JA, zorder=0, ls=(0, (1.0, 2.0)))
VY0, VY1, HX0, HX1 = 0.42, 6.68, 1.15, 13.05
for d in (-JD, JD):
    ax.plot([XC + d] * 2, [VY0, VY1], **JK)
    ax.plot([HX0, HX1], [YC + d] * 2, **JK)
b = 0.1
BK = dict(color=NEUTRAL, lw=0.5, alpha=JA, zorder=0)
for yend, sgn in ((VY1, 1), (VY0, -1)):
    ax.plot([XC - JD, XC - JD - b], [yend, yend - sgn * b], **BK)
    ax.plot([XC + JD, XC + JD + b], [yend, yend - sgn * b], **BK)
for xend, sgn in ((HX1, 1), (HX0, -1)):
    ax.plot([xend, xend - sgn * b], [YC + JD, YC + JD + b], **BK)
    ax.plot([xend, xend - sgn * b], [YC - JD, YC - JD - b], **BK)
for d in (-JD, JD):
    ax.add_patch(FancyBboxPatch((XC + d - 0.07, YC - 0.07), 0.14, 0.14, boxstyle='round,pad=0,rounding_size=0.04',
                                facecolor='white', edgecolor=NEUTRAL, lw=0.5, alpha=0.6, zorder=0))

# one dotted arc from each account to the next: A to B, B to C, C to D, D to A
def step(xy, xytext, rad):
    arrow(xy, xytext, color=GREY, lw=0.6, ls=(0, (1.2, 1.8)), rad=rad, ms=5, z=2, style='<|-|>')
step((HX0 - 0.25, YC - 0.4), (HX0 - 0.25, YC + 0.4), rad=0.35)
step((XC + 0.45, VY0 - 0.12), (XC - 0.45, VY0 - 0.12), rad=0.35)
step((HX1 + 0.25, YC + 0.4), (HX1 + 0.25, YC - 0.4), rad=0.35)
step((XC - 0.45, VY1 + 0.12), (XC + 0.45, VY1 + 0.12), rad=0.35)

# each account as its own panel file, cut from this drawing with the cross and the arcs between them left out
JOINS = [a for a in ax.get_children() if a not in _before]
save_panels(fig, 'Manuscript_Fig_1', {'A': (0.0, (YC + 0.02) * U, (XC - 0.02) * U, H * U),
                                      'B': (0.0, 0.0, (XC - 0.02) * U, (YC - 0.02) * U),
                                      'C': ((XC + 0.02) * U, 0.0, W * U, (YC - 0.02) * U),
                                      'D': ((XC + 0.02) * U, (YC + 0.02) * U, W * U, H * U)}, hide=JOINS)
plt.show()
print('Figure 1 drawn: A reflex chain, B processing hierarchy, C self-organized search, D element-level game, '
      'with the dotted cross and the four arcs between them, nothing computed from data')

panel files: Manuscript_Fig_1A, Manuscript_Fig_1B, Manuscript_Fig_1C, Manuscript_Fig_1D


Figure 1 drawn: A reflex chain, B processing hierarchy, C self-organized search, D element-level game, with the dotted cross and the four arcs between them, nothing computed from data
